# SmolVLM base: per-question prompt attention export

Enable Internet and GPU (one GPU or T4 x2). Attach your existing SmolVLM cache;
automatic discovery is supported. Push the implementation before cell 1.

All **16,817 test questions**, all **30 layers**. Query = last valid native prompt
token, including the assistant suffix, predicting the first answer token.
Softmax over all valid prompt keys including self; select 384 visual positions;
mean heads only. No generation, pruning, image-only normalization or sample mean.

**Initial batch 32 per GPU**; CUDA OOM halves it. FP32, no TF32. Two GPUs process
8,409/8,408 questions. Separate shards are committed every 100 successful batches,
at completion and on handled measurement errors. Progress distinguishes computed
from committed rows. A hard interruption can require repeating uncommitted work.

New/partial run: cells 1–7. Complete shards or existing export: **cells 1, 2, 6, 7**;
these do not download a model or scan features. Interrupting cell 5 leaves workers
running; rerun cell 5 to reconnect. Do not pull while workers run.

Keep the entire OUTPUT to resume. Changing GPU shard count or measurement identity
requires a new OUTPUT. Raw attention is ~739 MiB; shards + export + uncompressed ZIP
need ~2.2 GiB plus mapping/artifacts. No files are automatically deleted.


In [ ]:
# 1. Push Visualize first; this cell clones/pulls source without deleting outputs.
import os
from pathlib import Path
SOURCE_REPOSITORY = 'https://github.com/huyvanzzz/Log_Metrics.git'
SOURCE_BRANCH = 'smolvlm-attention'
SOURCE = Path('/kaggle/working/visualize-github-source')
OUTPUT = Path('/kaggle/working/smolvlm_prompt_attention_outputs')
RUN_MODE = 'auto'  # auto, measure, resume, export
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/kaggle/working/.cache/huggingface'
os.environ['PIP_CACHE_DIR'] = '/kaggle/working/.cache/pip'

"""Standard-library-only Git setup, embedded verbatim in both Kaggle notebooks."""
import hashlib
import importlib
import importlib.util
import io
import json
import os
from pathlib import Path
import subprocess
import sys
import zipfile


def ensure_no_workers(output_roots):
    for name in ('smolvlm_attention.runtime', 'smolvlm_attention.generated_runtime',
                 'dinov2_attention.runtime', 'smolvlm_probing.runtime',
                 'smolvlm_attention.sample_runtime', 'smolvlm_topk_benchmark.runtime'):
        module = sys.modules.get(name)
        if module is not None and any(process.poll() is None for process in getattr(module, '_PROCESSES', {}).values()):
            raise RuntimeError('A measurement worker is still running. Wait for it to finish before pulling source.')
    # The canonical probing output is shared by all notebooks. Also inspect it
    # after a kernel restart, when the new runtime is no longer in sys.modules.
    roots = set(map(Path, output_roots)) | {Path('/kaggle/working/smolvlm_probing_outputs'),
                                          Path('/kaggle/working/smolvlm_prompt_attention_outputs'),
                                          Path('/kaggle/working/smolvlm_topk_benchmark_outputs'),
                                          Path('/kaggle/working/smolvlm_topk_benchmark_outputs_batch64'),
                                          Path('/kaggle/working/smolvlm_topk_benchmark_outputs_batch64_two_waves')}
    for root in roots:
        for path in (Path(root) / 'logs').glob('*.pid.json'):
            pid = json.loads(path.read_text(encoding='utf-8'))['pid']
            try:
                os.kill(pid, 0)
            except (OSError, ProcessLookupError):
                continue
            raise RuntimeError(f'A measurement worker PID {pid} may still be running ({path}). Finish it before pulling.')


def sync_source(repository, branch, directory, packages=('smolvlm_inference', 'smolvlm_attention')):
    directory = Path(directory).resolve()
    def git(*args):
        return subprocess.check_output(['git', '-C', str(directory), *args], text=True).strip()
    if directory.exists():
        if not (directory / '.git').exists():
            raise RuntimeError(f'{directory} is not a Git checkout. Choose a new SOURCE directory.')
        if git('remote', 'get-url', 'origin').rstrip('/') != str(repository).rstrip('/'):
            raise RuntimeError('Existing source checkout has a different origin; choose a new SOURCE directory.')
        if git('status', '--porcelain', '--untracked-files=no'):
            raise RuntimeError('Source checkout has local changes. Preserve them before pulling; no files were reset.')
        if git('branch', '--show-current') != branch:
            raise RuntimeError('Checkout branch differs from SOURCE_BRANCH. Use the matching branch or a new SOURCE directory.')
        subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', 'origin', branch], check=True)
    else:
        directory.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', branch,
                        str(repository), str(directory)], check=True)
    # Keep the historical source_archive_sha256 field reproducible without embedding stale source.
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for package in packages:
            folder = directory / package
            if not folder.is_dir():
                raise RuntimeError(f'Missing source package after pull: {package}')
            for path in sorted(folder.iterdir()):
                if path.suffix in ('.py', '.txt'):
                    entry = zipfile.ZipInfo(path.relative_to(directory).as_posix(), (2026, 1, 1, 0, 0, 0))
                    entry.compress_type = zipfile.ZIP_DEFLATED
                    archive.writestr(entry, path.read_bytes())
    return dict(commit=git('rev-parse', 'HEAD'), sha256=hashlib.sha256(buffer.getvalue()).hexdigest())


def activate_source(directory, packages=('smolvlm_inference', 'smolvlm_attention')):
    directory = Path(directory).resolve()
    for package in packages:
        for path in (directory / package).glob('*.py'):
            Path(importlib.util.cache_from_source(str(path))).unlink(missing_ok=True)
    for name in list(sys.modules):
        if any(name == package or name.startswith(package + '.')
               for package in packages):
            del sys.modules[name]
    sys.path[:] = [str(directory)] + [entry for entry in sys.path if entry != str(directory)]
    importlib.invalidate_caches()

ensure_no_workers([OUTPUT, Path('/kaggle/working/attention_outputs'),
                   Path('/kaggle/working/generated_attention_outputs'),
                   Path('/kaggle/working/dinov2_attention_outputs')])
identity = sync_source(SOURCE_REPOSITORY, SOURCE_BRANCH, SOURCE)
SOURCE_COMMIT, SOURCE_SHA256 = identity['commit'], identity['sha256']
activate_source(SOURCE)
from smolvlm_attention.sample_runtime import resolve_run_mode
BOOT_MODE = resolve_run_mode(OUTPUT, RUN_MODE)
if BOOT_MODE != 'export':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers',
                    '-r', str(SOURCE/'smolvlm_attention/sample_requirements.txt')], check=True)
    import transformers, tokenizers
    if transformers.__version__ != '4.46.3' or tokenizers.__version__ != '0.20.4':
        raise RuntimeError('Restart kernel after dependency installation, then rerun cell 1.')
RUN_CONFIG = OUTPUT/'sample_run.json'
print(dict(mode=BOOT_MODE, output=str(OUTPUT), commit=SOURCE_COMMIT))


In [ ]:
# 2. Defaults work without editing when one matching cache is attached.
BATCH_SIZE = 32  # Per GPU; only CUDA OOM reduces it.
GPU_COUNT = None  # Auto: up to two GPUs. Resume must retain original topology.
CACHE_INPUT = ''  # Set only if multiple matching mounted caches exist.
PREPARED_INPUT = None  # Optional existing prepared-data directory; otherwise pinned HF.
from smolvlm_attention.sample_runtime import prepare_run, launch_workers, wait_workers
from smolvlm_attention.sample_store import finalize_export, open_prompt_attention, bundle_export
print(dict(initial_batch_per_gpu=BATCH_SIZE, mode=BOOT_MODE))


In [ ]:
# 3. Validate cache once and tokenize every test prompt without answer/truncation.
if BOOT_MODE != 'export':
    RUN_CONFIG = prepare_run(OUTPUT, mode=BOOT_MODE, batch_size=BATCH_SIZE,
        gpu_count=GPU_COUNT, cache_input_dir=CACHE_INPUT, prepared_dir=PREPARED_INPUT,
        source_commit=SOURCE_COMMIT, source_sha256=SOURCE_SHA256)
print(RUN_CONFIG)


In [ ]:
# 4. Each worker strict-loads base and checks native eager/batch parity before measuring.
if BOOT_MODE != 'export':
    launch_workers(RUN_CONFIG)


In [ ]:
# 5. Live tqdm per GPU + total. Interrupting does not cancel workers.
if BOOT_MODE != 'export':
    wait_workers(RUN_CONFIG)


In [ ]:
# 6. Verify shard checksums and merge in original test order. No model/cache needed.
EXPORT = finalize_export(RUN_CONFIG)
RESULTS = open_prompt_attention(EXPORT)
print(dict(shape=RESULTS.attention.shape, dtype=str(RESULTS.attention.dtype),
           first_sample=RESULTS.samples[0]['sample_id'], export=str(EXPORT)))
print('Example: sample 0, layer 21, first eight visual tokens:', RESULTS.attention[0, 21, :8])


In [ ]:
# 7. ZIP only portable export (no checkpoint/cache/shards). Extract before memory mapping.
ZIP = bundle_export(EXPORT)
from IPython.display import FileLink, display
display(FileLink(str(ZIP)))
for name in ('attention.npy', 'samples.jsonl', 'metadata.json', 'manifest.json', 'README.md'):
    display(FileLink(str(EXPORT/name)))
